# Benchmark détection + modélisation des chirps

Ce notebook compare `bat_chirp_annotations.json` avec le détecteur SNR/blob et `process_full_spectrum()` du script Python choisi.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from tkinter import Tk, filedialog
from benchmark import run_benchmark


## Choisir le JSON d'annotation et le script de traitement

In [ ]:
root = Tk()
root.withdraw()
root.attributes('-topmost', True)

annotation_json = filedialog.askopenfilename(
    title='Sélectionner bat_chirp_annotations.json',
    filetypes=[('JSON', '*.json'), ('Tous les fichiers', '*.*')],
)
analysis_py = filedialog.askopenfilename(
    title='Sélectionner le script Python de traitement',
    filetypes=[('Python', '*.py'), ('Tous les fichiers', '*.*')],
)
root.destroy()

print('Annotations :', annotation_json)
print('Traitement  :', analysis_py)


## Lancer le benchmark

Les paramètres ci-dessous correspondent aux valeurs actuellement utilisées par `process_wav_file(..., detector='snr_blob')`.

In [ ]:
result = run_benchmark(
    annotation_json=annotation_json,
    analysis_py=analysis_py,
    min_iou=0.05,
    max_center_error_ms=4.0,
    verbose=True,
)


## Résumé global

In [ ]:
import pandas as pd
pd.Series(result.summary)


## Résultats par WAV

In [ ]:
result.files

## Résultats par chirp

`failure_stage` distingue une erreur de détection d'un échec de modélisation.

In [ ]:
cols = [
    'relative_path', 'chirp_id', 'detected', 'model_success', 'failure_stage',
    'detection_center_error_ms', 'detection_iou',
    'mae_khz', 'rmse_khz', 'p95_abs_error_khz', 'coverage',
    'start_error_ms', 'end_error_ms'
]
result.chirps[[c for c in cols if c in result.chirps.columns]]

## Chirps en échec uniquement

In [ ]:
result.chirps[result.chirps['failure_stage'].fillna('') != '']

## Sauvegarder les résultats

In [ ]:
output_dir = Path(annotation_json).parent / 'benchmark_results'
result.save_csv(output_dir)
print('Résultats sauvegardés dans :', output_dir)
